# Task 5d (C2): session-level leakage
Runtime → Change runtime type → **T4 GPU**. Upload `sessions.py`, `capture_sessions.py`, `session_attack.py` and the CURRENT `secret_ushape.py` (with `save_adapters`) to `/content`.

Needs the saved adapters of `{SAVE}/runs/H4_T2_seed0/` and `{SAVE}/runs/H2_T2_seed1/` (see `run_task5_colab.ipynb`).
Per split: 600 templated sessions (400 train / 200 test), attacker A on the public model, capture public + protected layer-H states with the KV cache, attackers per-turn / union / vote (upper bounds) / realistic (no alignment) / template-token / B, and the unprotected control by position bucket. Runs λa = 0 and 0.03 for (4,2) and (2,2); attacker A is trained once per split and reused. Run time on a T4 not measured yet.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install -q datasets
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/multisplit'
OUT = SAVE + '/task5d'
import os
os.makedirs(OUT, exist_ok=True)
RUNS = {'H4_T2': SAVE + '/runs/H4_T2_seed0', 'H2_T2': SAVE + '/runs/H2_T2_seed1'}
LAS = ['0', '0.03']
for k, d in RUNS.items():
    for la in LAS:
        f = f'{d}/adapters_lp0.1_la{la}.pt'
        print(k, la, f, os.path.exists(f))

## Code test (CPU, tiny random model) — optional
Includes the self-test of the session metrics (an attacker of known quality).

In [ ]:
!python sessions.py --smoke --n 30 --out /content/smoke_sessions
!python session_attack.py --smoke --out /content/smoke_5d

## Session statistics (real tokenizer)

In [ ]:
!python sessions.py --out {OUT}/sessions

## Session attack: one cell per split, λa = 0 then 0.03 (results and the cached attacker A on Drive)

In [ ]:
for la in LAS:
    f = f"{RUNS['H4_T2']}/adapters_lp0.1_la{la}.pt"
    !python session_attack.py --adapters {f} --out {OUT}/H4_T2_la{la}

In [ ]:
for la in LAS:
    f = f"{RUNS['H2_T2']}/adapters_lp0.1_la{la}.pt"
    !python session_attack.py --adapters {f} --out {OUT}/H2_T2_la{la}

## Download

In [ ]:
!cd {OUT} && zip -qr /content/task5d_results.zip .
from google.colab import files
files.download('/content/task5d_results.zip')